In [1]:
!pip install pandas scikit-learn streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 66.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 72.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [8]:

import os

print(os.listdir("/content"))


['.config', 'movies.csv', 'sample_data']


In [2]:

import pandas as pd

movies_data = {
    "title": [
        "The Avengers",
        "Iron Man",
        "Guardians of the Galaxy",
        "Titanic",
        "The Notebook",
        "Avatar",
        "Interstellar",
        "The Dark Knight",
        "Spider-Man",
        "Doctor Strange"
    ],
    "genres": [
        "Action Adventure Sci-Fi",
        "Action Adventure Sci-Fi",
        "Action Adventure Sci-Fi",
        "Romance Drama",
        "Romance Drama",
        "Action Adventure Sci-Fi",
        "Adventure Drama Sci-Fi",
        "Action Crime Drama",
        "Action Adventure Sci-Fi",
        "Action Adventure Fantasy"
    ]
}

movies = pd.DataFrame(movies_data)
movies.to_csv("movies.csv", index=False)

movies


,title,genres
0,The Avengers,Action Adventure Sci-Fi
1,Iron Man,Action Adventure Sci-Fi
2,Guardians of the Galaxy,Action Adventure Sci-Fi
3,Titanic,Romance Drama
4,The Notebook,Romance Drama
5,Avatar,Action Adventure Sci-Fi
6,Interstellar,Adventure Drama Sci-Fi
7,The Dark Knight,Action Crime Drama
8,Spider-Man,Action Adventure Sci-Fi
9,Doctor Strange,Action Adventure Fantasy


In [3]:

movies = pd.read_csv("movies.csv")

print(movies.head())
print("Total movies:", len(movies))


                     title                   genres
0             The Avengers  Action Adventure Sci-Fi
1                 Iron Man  Action Adventure Sci-Fi
2  Guardians of the Galaxy  Action Adventure Sci-Fi
3                  Titanic            Romance Drama
4             The Notebook            Romance Drama
Total movies: 10


In [4]:

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
genre_matrix = vectorizer.fit_transform(movies["genres"])

print(vectorizer.get_feature_names_out())
print(genre_matrix.toarray())


['action' 'adventure' 'crime' 'drama' 'fantasy' 'fi' 'romance' 'sci']
[[1 1 0 0 0 1 0 1]
 [1 1 0 0 0 1 0 1]
 [1 1 0 0 0 1 0 1]
 [0 0 0 1 0 0 1 0]
 [0 0 0 1 0 0 1 0]
 [1 1 0 0 0 1 0 1]
 [0 1 0 1 0 1 0 1]
 [1 0 1 1 0 0 0 0]
 [1 1 0 0 0 1 0 1]
 [1 1 0 0 1 0 0 0]]


In [5]:

from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(genre_matrix)

print(similarity.round(2))


[[1.   1.   1.   0.   0.   1.   0.75 0.29 1.   0.58]
 [1.   1.   1.   0.   0.   1.   0.75 0.29 1.   0.58]
 [1.   1.   1.   0.   0.   1.   0.75 0.29 1.   0.58]
 [0.   0.   0.   1.   1.   0.   0.35 0.41 0.   0.  ]
 [0.   0.   0.   1.   1.   0.   0.35 0.41 0.   0.  ]
 [1.   1.   1.   0.   0.   1.   0.75 0.29 1.   0.58]
 [0.75 0.75 0.75 0.35 0.35 0.75 1.   0.29 0.75 0.29]
 [0.29 0.29 0.29 0.41 0.41 0.29 0.29 1.   0.29 0.33]
 [1.   1.   1.   0.   0.   1.   0.75 0.29 1.   0.58]
 [0.58 0.58 0.58 0.   0.   0.58 0.29 0.33 0.58 1.  ]]


In [6]:

def recommend_movies(movie_title, number=5):
    movie_index = movies[
        movies["title"] == movie_title
    ].index[0]

    scores = list(enumerate(similarity[movie_index]))

    scores = sorted(
        scores,
        key=lambda item: item[1],
        reverse=True
    )

    scores = [
        item for item in scores
        if item[0] != movie_index
    ]

    recommendations = []

    for index, score in scores[:number]:
        recommendations.append({
            "Movie": movies.iloc[index]["title"],
            "Genres": movies.iloc[index]["genres"],
            "Similarity": round(float(score), 2)
        })

    return pd.DataFrame(recommendations)


In [7]:

recommend_movies("The Avengers")


,Movie,Genres,Similarity
0,Iron Man,Action Adventure Sci-Fi,1.00
1,Guardians of the Galaxy,Action Adventure Sci-Fi,1.00
2,Avatar,Action Adventure Sci-Fi,1.00
3,Spider-Man,Action Adventure Sci-Fi,1.00
4,Interstellar,Adventure Drama Sci-Fi,0.75
